# Iceberg Drift Prediction - Inference Demo

This notebook demonstrates how to use a trained model for inference.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve() / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from iceberg_drift.utils import DriftPredictor, load_model_for_inference
from iceberg_drift.utils.visualization import plot_trajectory, create_trajectory_animation
from iceberg_drift.utils.export import export_to_geojson, export_route_for_navigation

## 1. Load Trained Model

**Note**: You need a trained model checkpoint. Run the training script first:
```bash
python scripts/train.py --model-type pinn --epochs 50
```
Then update the path below.

In [ ]:
# Update this path to your trained model
model_path = "../output/checkpoints/best_model.pt"

# For demo, we'll create a dummy predictor if model doesn't exist
import os
if not os.path.exists(model_path):
    print(f"Model not found at {model_path}")
    print("Please train a model first using scripts/train.py")
    # Create a minimal demo with untrained model
    from iceberg_drift.models import PINNDriftModel
    import torch
    
    dummy_model = PINNDriftModel(input_dim=50, output_dim=2, prediction_horizon=12)
    torch.save({'model_state_dict': dummy_model.state_dict(), 'input_dim': 50}, model_path)
    print("Created dummy model for demo")

In [ ]:
# Load predictor
predictor = load_model_for_inference(
    model_path,
    prediction_horizon_hours=72,
    device="auto",
)

print("Predictor loaded successfully")

## 2. Prepare Environmental Forcing Data

In practice, you would get this from:
- ERA5/ERA5-Land for wind forecasts
- Copernicus Marine Service (CMEMS) for ocean current forecasts
- National weather services

For this demo, we'll generate synthetic forcing.

In [ ]:
# Initial iceberg position
init_lat = -65.5  # Weddell Sea
init_lon = -45.0
iceberg_length = 3000  # meters
iceberg_width = 1500

# Historical data (last 24 hours at 6-hourly = 4 steps)
n_hist = 4
base_time = pd.Timestamp.now().floor('6H')
hist_times = pd.date_range(base_time - pd.Timedelta(hours=24), base_time, freq='6H')

# Historical positions (assume we have tracking data)
hist_lats = np.array([-65.5, -65.48, -65.45, -65.42])
hist_lons = np.array([-45.0, -44.95, -44.90, -44.85])

# Historical environmental forcing
np.random.seed(42)
hist_current_u = np.random.normal(0.05, 0.05, n_hist)
hist_current_v = np.random.normal(-0.02, 0.05, n_hist)
hist_wind_u = np.random.normal(5, 8, n_hist)
hist_wind_v = np.random.normal(-3, 8, n_hist)

print("Historical data prepared")
print(f"Times: {hist_times}")
print(f"Positions: {list(zip(hist_lats, hist_lons))}")

## 3. Run Prediction

In [ ]:
# Predict trajectory
result = predictor.predict(
    timestamps=hist_times,
    latitudes=hist_lats,
    longitudes=hist_lons,
    current_u=hist_current_u,
    current_v=hist_current_v,
    wind_u=hist_wind_u,
    wind_v=hist_wind_v,
    iceberg_length=iceberg_length,
    iceberg_width=iceberg_width,
)

print("Prediction complete!")
print(f"Initial: ({result['init_lat']:.3f}, {result['init_lon']:.3f})")
print(f"Final: ({result['latitudes'][-1]:.3f}, {result['longitudes'][-1]:.3f})")
print(f"Steps: {len(result['latitudes']) - 1}")

## 4. Visualize Prediction

In [ ]:
# Plot trajectory
fig = plot_trajectory(
    latitudes=result['latitudes'],
    longitudes=result['longitudes'],
    timestamps=result['timestamps'],
    title=f"Iceberg Drift Prediction (72h)",
    projection="polar",
    color_by=np.arange(len(result['latitudes'])),
    color_label="Time Step",
    output_path="../output/demo_trajectory.png",
)

plt.show()

In [ ]:
# Compare with physics-only baseline if available
if 'physics_latitudes' in result:
    fig = plot_trajectory_comparison(
        true_latitudes=result['latitudes'],  # Using ML as reference
        true_longitudes=result['longitudes'],
        pred_latitudes=result['physics_latitudes'],
        pred_longitudes=result['physics_longitudes'],
        timestamps=result['timestamps'],
        title="ML vs Physics Baseline",
        output_path="../output/demo_comparison.png",
    )
    plt.show()

## 5. Export Results

In [ ]:
# Export to various formats
trajectories = {
    "predicted": {
        "latitudes": result["latitudes"],
        "longitudes": result["longitudes"],
        "timestamps": result["timestamps"],
        "velocities_u": result["velocities_u"],
        "velocities_v": result["velocities_v"],
    }
}

if "uncertainty_km" in result:
    trajectories["predicted"]["uncertainty_km"] = result["uncertainty_km"]

# GeoJSON for web mapping
export_to_geojson(
    trajectories,
    "../output/demo_trajectory.geojson",
    properties={
        "model": model_path,
        "init_lat": init_lat,
        "init_lon": init_lon,
        "iceberg_length_m": iceberg_length,
        "iceberg_width_m": iceberg_width,
    },
)

# GPX for navigation
export_route_for_navigation(
    trajectories["predicted"],
    "../output/demo_route.gpx",
    format="gpx",
)

print("Exports complete!")

## 6. Create Animation

In [ ]:
# Create animated trajectory (requires cartopy + ffmpeg)
try:
    anim = create_trajectory_animation(
        trajectories={"Predicted": (result["latitudes"], result["longitudes"])} ,
        timestamps=result["timestamps"],
        title="Iceberg Drift Prediction",
        fps=2,
        interval=500,
        output_path="../output/demo_animation.mp4",
    )
    print("Animation created!")
except Exception as e:
    print(f"Animation failed (need ffmpeg): {e}")